# Pairs Trading Project: Data Download

This notebook downloads daily S&P 500 price data and checks it **before** later analysis.

## How we handle missing or unusable ticker files

1. Keep only companies that appeared in the S&P 500 during our study period. The source file begins in 1996, but we do not need every ticker from that full history.
2. Change ticker punctuation to Yahoo Finance's format, for example `BRK.B` becomes `BRK-B`.
3. Try each download twice, because a temporary internet problem should not permanently reject a company.
4. Check that the result contains at least one usable date and closing price.
5. Save every nonempty price history, even when it is short. Never invent prices for a delisted or unavailable company.
6. Save a report for every ticker and a separate list of valid tickers. Later notebooks should use the valid list instead of reading every file in the folder.

> A missing delisted ticker is recorded and skipped. Replacing it with a present-day company or filling it with fake values would bias the analysis. A minimum-history rule, such as 252 days, will be applied later when pairs are selected—not while raw data is downloaded.

In [1]:
import pandas as pd
import yfinance as yf
import time
from pathlib import Path

print("Setup is working!")

Setup is working!


In [2]:
# Settings are kept together so they are easy to find and change.
START_DATE = "2015-01-01"
END_DATE = "2026-08-25"
MAX_ATTEMPTS = 2

# This works whether Jupyter starts in the project folder or notebooks folder.
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data" / "historical"
REPORT_DIR = PROJECT_DIR / "data" / "reports"
DATA_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
def yahoo_ticker(original_ticker):
    """Convert the source ticker to the simple format Yahoo expects."""
    ticker = original_ticker.strip().split()[0]
    return ticker.replace(".", "-")


def download_and_check(original_ticker, start, end):
    """Download one ticker and return a small result record."""
    ticker = yahoo_ticker(original_ticker)
    last_error = ""

    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            data = yf.download(
                ticker,
                start=start,
                end=end,
                interval="1d",
                auto_adjust=False,
                progress=False,
            )

            # yfinance may return two levels of column names. We need only
            # the first level: Open, High, Low, Close, Adj Close, Volume.
            if data.columns.nlevels > 1:
                data.columns = data.columns.get_level_values(0)

            data = data.reset_index()

            if data.empty or "Date" not in data or "Close" not in data:
                last_error = "Yahoo returned no price rows"
            else:
                data = data.dropna(subset=["Date", "Close"])
                data = data.drop_duplicates(subset="Date").sort_values("Date")

                data.to_csv(DATA_DIR / f"{ticker}.csv", index=False)
                return {
                    "original_ticker": original_ticker,
                    "yahoo_ticker": ticker,
                    "status": "saved",
                    "rows": len(data),
                    "message": "Passed all checks",
                }

        except Exception as error:
            last_error = str(error)

        if attempt < MAX_ATTEMPTS:
            time.sleep(1)

    return {
        "original_ticker": original_ticker,
        "yahoo_ticker": ticker,
        "status": "unavailable",
        "rows": 0,
        "message": last_error,
    }


In [4]:
import pandas as pd

url = "https://raw.githubusercontent.com/chinobing/historical_sp500_constituents/main/sp_500_historical_components.csv"

sp500_hist = pd.read_csv(url)

print(sp500_hist.head())
print(sp500_hist.columns)

         date                                            tickers
0  1996-01-02  AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD...
1  1996-01-03  AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD...
2  1996-01-04  AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD...
3  1996-01-10  AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD...
4  1996-01-11  AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD...
Index(['date', 'tickers'], dtype='str')


In [5]:
# The source contains membership back to 1996. For downloading, make one
# combined list of companies seen during our study period. Later analysis
# must still check whether each company was a member on each particular date.
sp500_hist["date"] = pd.to_datetime(sp500_hist["date"])
period_membership = sp500_hist[
    (sp500_hist["date"] >= START_DATE) & (sp500_hist["date"] <= END_DATE)
]

original_tickers = set()
for ticker_text in period_membership["tickers"].dropna():
    original_tickers.update(ticker_text.split(","))

# Keep one original symbol for each normalized Yahoo symbol.
ticker_lookup = {}
for original_ticker in sorted(original_tickers):
    ticker_lookup.setdefault(yahoo_ticker(original_ticker), original_ticker)

tickers_to_download = list(ticker_lookup.values())
print(f"Tickers to check: {len(tickers_to_download)}")

Tickers to check: 737


In [6]:
# Download, validate, and remember what happened to every ticker.
results = []
for number, ticker in enumerate(tickers_to_download, start=1):
    result = download_and_check(ticker, START_DATE, END_DATE)
    results.append(result)
    print(
        f"{number}/{len(tickers_to_download)} "
        f"{result['yahoo_ticker']}: {result['status']} ({result['rows']} rows)"
    )

download_report = pd.DataFrame(results)
download_report.to_csv(REPORT_DIR / "download_report.csv", index=False)

valid_tickers = download_report.loc[
    download_report["status"] == "saved", ["yahoo_ticker", "rows"]
].sort_values("yahoo_ticker")
valid_tickers.to_csv(REPORT_DIR / "valid_tickers.csv", index=False)

print("\nDownload summary:")
print(download_report["status"].value_counts())
print(f"\nValid ticker list saved to: {REPORT_DIR / 'valid_tickers.csv'}")
download_report.head()

1/737 A: saved (2927 rows)
2/737 AAL: saved (2927 rows)
3/737 AAP: saved (2927 rows)
4/737 AAPL: saved (2927 rows)
5/737 ABBV: saved (2927 rows)


$ABC: possibly delisted; no timezone found

1 Failed download:
['ABC']: possibly delisted; no timezone found
$ABC: possibly delisted; no timezone found

1 Failed download:
['ABC']: possibly delisted; no timezone found


6/737 ABC: unavailable (0 rows)


$ABMD: possibly delisted; no timezone found

1 Failed download:
['ABMD']: possibly delisted; no timezone found
$ABMD: possibly delisted; no timezone found

1 Failed download:
['ABMD']: possibly delisted; no timezone found


7/737 ABMD: unavailable (0 rows)
8/737 ABNB: saved (1431 rows)
9/737 ABT: saved (2927 rows)
10/737 ACGL: saved (2927 rows)
11/737 ACN: saved (2927 rows)
12/737 ADBE: saved (2927 rows)
13/737 ADI: saved (2927 rows)
14/737 ADM: saved (2927 rows)
15/737 ADP: saved (2927 rows)


...

$XEC: possibly delisted; no timezone found

1 Failed download:
['XEC']: possibly delisted; no timezone found
$XEC: possibly delisted; no timezone found

1 Failed download:
['XEC']: possibly delisted; no timezone found


725/737 XEC: unavailable (0 rows)
726/737 XEL: saved (2927 rows)


$XLNX: possibly delisted; no timezone found

1 Failed download:
['XLNX']: possibly delisted; no timezone found
$XLNX: possibly delisted; no timezone found

1 Failed download:
['XLNX']: possibly delisted; no timezone found


727/737 XLNX: unavailable (0 rows)
728/737 XOM: saved (2927 rows)
729/737 XRAY: saved (2927 rows)
730/737 XRX: saved (2927 rows)
731/737 XYL: saved (2927 rows)
732/737 XYZ: saved (2704 rows)
733/737 YUM: saved (2927 rows)
734/737 ZBH: saved (2927 rows)
735/737 ZBRA: saved (2927 rows)
736/737 ZION: saved (2927 rows)
737/737 ZTS: saved (2927 rows)

Download summary:
status
saved          616
unavailable    121
Name: count, dtype: int64

Valid ticker list saved to: /home/lenovo/projects/pairs_trading_project/data/reports/valid_tickers.csv


,original_ticker,yahoo_ticker,status,rows,message
0,A,A,saved,2927,Passed all checks
1,AAL,AAL,saved,2927,Passed all checks
2,AAP,AAP,saved,2927,Passed all checks
3,AAPL,AAPL,saved,2927,Passed all checks
4,ABBV,ABBV,saved,2927,Passed all checks
